# ML-04: Search Intelligence Data Contract

## 1. Five contract answers

1. **Grain:** raw data is one report date × pseudonymized client × content item. The model frame is one client × content item with a first-half feature window and a disjoint second-half outcome window.
2. **Table:** `FlyRank/internship-warehouse`, release v20260703, `fact_content_daily_performance/month=2026-03/data_0.parquet`. No joins, query-level rows or product flags are required.
3. **Time:** March 1–15 supplies features. The decision cutoff is the end of March 15. March 16–31 supplies observed CTR for evaluation only. These are retrospective complete windows, not a claim of zero-lag production availability. June is not used for development.
4. **Target/action:** predict the later observed CTR (percentage points) to assess whether a small model improves on persistence; use a separate descriptive CTR-vs-position gap to rank pages for search-intent/snippet review. A score is not an observed actionability label.
5. **Exclusion:** exclude every March 16–31 measurement from the five inputs. Also exclude IDs as features, all query-table aggregates, trend labels, and GA4 fields whose missing availability could be confused with zero traffic.

Only rows with `gsc_data_available IS TRUE` are aggregated. Feature eligibility requires all 15 first-half dates available, at least 500 first-half impressions and an impression-weighted valid average position between 1 and 20. Evaluation additionally requires all 16 outcome dates and at least 500 outcome impressions. That future condition is used only to decide whether CTR can be evaluated; it is not a filter on the operational review queue.

## 2. Field roles and availability

| Role | Fields / transformation | Available when / reason |
|---|---|---|
| Feature 1 | `past_ctr = 100 × SUM(gsc_clicks)/SUM(gsc_impressions)` | March 1–15 observed search totals, known after the cutoff/reporting lag |
| Feature 2 | `log_impressions = log(1 + SUM(gsc_impressions))` | Only the same first-half window |
| Feature 3 | `mean_position` | First-half impression-weighted `gsc_avg_position`; unknown/nonpositive values excluded from position numerator and denominator |
| Feature 4 | `impression_cv` | Standard deviation / mean of first-half daily impressions; describes instability known at cutoff |
| Feature 5 | `zero_click_fraction` | Fraction of first-half impression-active days with zero clicks |
| Label | `future_ctr`, `future_impressions`, `future_available_days` | March 16–31; target, evaluation reliability/weights only |
| Context | `report_date`, `client_hash_id`, `content_hash_id`, availability flags, past counts, position band | Date boundaries, grouping, splitting, eligibility and explanation only |
| Excluded | All remaining daily columns, `gsc_sum_position`, query-table fields, external metadata and any recreated product flags | Not necessary for this five-feature question; no uncontrolled joins or ambiguous time windows |

## 3. Exactly three verification queries

These queries run on the March partition. The third checks availability explicitly with `IS TRUE` and measures missingness in the fields used. Feature aggregation below is a transformation, separate from these three verification queries.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'work/ctr_study.py').is_file())
sys.path.insert(0,str(ROOT/'work'))
import ctr_study as study
import numpy as np
import pandas as pd
from IPython.display import display
con=study.connection()
for name, sql in study.VERIFICATION_QUERIES.items():
    print(name.upper())
    print(sql)
    result=con.sql(sql).df()
    display(result)
    if name=='grain': assert result.iloc[0,0]==0
    if name=='count_and_span':
        assert result.iloc[0]['rows']==9841378
        assert result.iloc[0]['missing_keys']==0
con.close()


GRAIN
SELECT COUNT(*) AS duplicate_grain_groups FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) n
        FROM daily GROUP BY 1,2,3 HAVING COUNT(*) > 1)


,duplicate_grain_groups
0,0


COUNT_AND_SPAN
SELECT COUNT(*) AS rows, MIN(report_date) AS first_date,
        MAX(report_date) AS last_date, COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        COUNT(*) FILTER (WHERE report_date IS NULL OR client_hash_id IS NULL
          OR content_hash_id IS NULL) AS missing_keys FROM daily


,rows,first_date,last_date,clients,content_items,missing_keys
0,9841378,2026-03-01,2026-03-31,55,331437,0


AVAILABILITY
SELECT COUNT(*) AS all_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE) AS both_available,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND
          (gsc_impressions IS NULL OR gsc_clicks IS NULL)) AS missing_search_counts,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND gsc_impressions>0
          AND (gsc_avg_position IS NULL OR gsc_avg_position<=0)) AS unknown_position_days
        FROM daily


,all_rows,gsc_available,ga4_available,both_available,missing_search_counts,unknown_position_days
0,9841378,3611061,413966,364347,0,163189


The raw partition has **9,841,378 rows**, **55 clients**, **331,437 content items**, no missing keys and no duplicate grain groups. Search data is available on **3,611,061 rows**; GA4 on **413,966**; both on **364,347**. Search-available rows have no null impression/click counts. **163,189** search-available, impression-positive rows lack a positive position: these do not become rank zero. Availability differs substantially, so the presence of daily rows must not be interpreted as complete measurement coverage.

In [2]:
frame=study.make_frame('march')
feature_frame=frame[study.FEATURES]
assert feature_frame.shape[1]==5
display(feature_frame.head(5).round(5))
display(feature_frame.agg(['count','min','max']).round(5))
display(pd.DataFrame({'measure':['feature-eligible pages','evaluation-eligible pages','feature clients'],
                     'value':[len(frame),int(frame.evaluation_eligible.sum()),frame.client_hash_id.nunique()]}))
# Stable client folds: 0 final test, 1 development validation, 2-4 training.
import hashlib
frame['fold']=frame.client_hash_id.map(lambda x:int(hashlib.sha256(('ctr-v1:'+x).encode()).hexdigest()[:8],16)%5)
evaluated=frame.loc[frame.evaluation_eligible].copy()
train=evaluated.loc[evaluated.fold>=2]
valid=evaluated.loc[evaluated.fold==1]
assert not set(train.client_hash_id)&set(valid.client_hash_id)
print('Final-test fold 0 is excluded from this experiment.')


,past_ctr,log_impressions,mean_position,impression_cv,zero_click_fraction
0,0.86366,7.39142,7.54596,0.72722,0.46667
1,0.21398,7.24637,9.07846,0.46955,0.80000
2,0.00000,6.84588,9.41640,0.93498,1.00000
3,0.46729,6.75344,9.31776,0.94653,0.80000
4,0.46729,6.75344,9.31776,0.94653,0.80000


,past_ctr,log_impressions,mean_position,impression_cv,zero_click_fraction
count,28862.00000,28862.00000,28862.00000,28862.00000,28862.0
min,0.00000,6.21661,1.00000,0.05926,0.0
max,7.31707,11.87182,19.99801,3.37139,1.0


,measure,value
0,feature-eligible pages,28862
1,evaluation-eligible pages,25772
2,feature clients,29


Final-test fold 0 is excluded from this experiment.


### Deliberate leakage experiment

Fit a standardized linear model on the five first-half features and score only validation clients. Then deliberately append the **exact later CTR target** as a sixth input. The near-perfect result is invalid: it supplies the answer. Delete this column and recompute the honest error on the identical rows. Final-test clients are not used in this demonstration.

In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,r2_score
Xtr=train[study.FEATURES].copy(); Xv=valid[study.FEATURES].copy()
ytr=train.future_ctr; yv=valid.future_ctr
honest=make_pipeline(StandardScaler(),LinearRegression()).fit(Xtr,ytr)
honest_pred=np.clip(honest.predict(Xv),0,100)
Xtr['LEAK_future_ctr']=ytr; Xv['LEAK_future_ctr']=yv
leaky=make_pipeline(StandardScaler(),LinearRegression()).fit(Xtr,ytr)
leaky_pred=leaky.predict(Xv)
del Xtr['LEAK_future_ctr']; del Xv['LEAK_future_ctr']
restored=make_pipeline(StandardScaler(),LinearRegression()).fit(Xtr,ytr)
restored_pred=np.clip(restored.predict(Xv),0,100)
assert np.allclose(honest_pred,restored_pred)
assert Xtr.columns.tolist()==study.FEATURES
display(pd.DataFrame([{'run':name,'MAE_pp':mean_absolute_error(yv,p),'R2':r2_score(yv,p)}
    for name,p in [('honest five features',honest_pred),('INVALID: target supplied',leaky_pred),
                   ('leak deleted',restored_pred)]]).round(8))
study.save_json('data_contract_metrics.json',{'rows':9841378,'features':study.FEATURES,
    'feature_pages':len(frame),'evaluation_pages':len(evaluated),
    'honest_validation_mae_pp':float(mean_absolute_error(yv,restored_pred)),
    'invalid_leaky_mae_pp':float(mean_absolute_error(yv,leaky_pred))})


,run,MAE_pp,R2
0,honest five features,0.134978,0.548467
1,INVALID: target supplied,0.000000,1.000000
2,leak deleted,0.134978,0.548467


## 4. Limits and output

This is one month of an unbalanced warehouse panel, not a representative sample of all clients or seasons. Complete-window and minimum-volume filters exclude young, low-volume and weakly measured pages. A position-weighted average cannot control query mix, device, country or search-result layout. The model cannot establish that metadata changes cause clicks; no human actionability labels are available.

**Output:** an explainable queue for human review plus a separate evaluation of later observed CTR error. The ranking is decision support, not an automatic editing system.

## 5. Self-check

- Three verification queries and their outputs are above; the grain, dates and availability are checked.
- Exactly five honest inputs, all preceding the outcome window; the leaked column is removed.
- Raw data and local queues stay outside Git. Only aggregate receipts and minimal feature examples are displayed.
- This notebook must be run top to bottom using `work/REPRODUCE.md` before submission.

Sources: the assignment brief, local `docs/data-dictionary.md`, and FlyRank warehouse release v20260703.